# CMRxRecon 2023 → Google Drive

Baixa o dataset **CMRxRecon 2023** (cine SAX + 2CH/3CH/4CH com segmentações) do Synapse direto para o seu Google Drive e extrai **só** o necessário para o DeepMesh.

**Antes de rodar**
1. Crie uma conta em [synapse.org](https://www.synapse.org), abra o dataset [syn51471091](https://www.synapse.org/#!Synapse:syn51471091/wiki/) e aceite os termos de uso, se pedir.
2. Em *Account Settings → Personal Access Tokens*, gere um token com **View** e **Download**.
3. Aqui no Colab, clique no ícone de **chave (Secrets)** na barra lateral, crie um segredo chamado `SYNAPSE_TOKEN` com o token e ative o acesso para este notebook. **Não cole o token no código.**
4. Use um ambiente **sem GPU** (*Ambiente de execução → Alterar tipo → CPU*): o download não precisa de GPU e assim você não gasta cota.

Se a sessão cair, é só rodar as células de novo: o download continua de onde parou.

## 1. Drive e configurações

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DEST = '/content/drive/MyDrive/datasets/CMRxRecon2023'   # onde tudo fica no Drive
PARTS_DIR = os.path.join(DEST, 'parts')                  # partes .tar.gz baixadas
OUT_DIR = os.path.join(DEST, 'extracted')                # arquivos extraídos (só o necessário)
os.makedirs(PARTS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# IDs no Synapse (estrutura listada pela API pública)
TRAIN_FOLDER = 'syn53192104'        # ChallengeData_ : ChallengeData.tar.gz-part-00..60 + text-files
AFTER_FOLDER = 'syn53184437'        # ChallengeData_AfterCompetition_ : validation (12 partes) + test (37 partes)

## 2. Login no Synapse (token nos Secrets do Colab)

In [ ]:
!pip -q install synapseclient
import synapseclient
from google.colab import userdata

syn = synapseclient.Synapse(silent=True)
syn.login(authToken=userdata.get('SYNAPSE_TOKEN'))
print('logado como', syn.getUserProfile()['userName'])

## 3. Ler a documentação e ver o tamanho de cada parte

Nada grande é baixado aqui: só os arquivos de texto e os metadados. **Leia a saída antes de seguir** — ela diz quanto espaço o download vai ocupar e como o dataset está organizado.

In [ ]:
import json

def children(parent):
    return list(syn.getChildren(parent, includeTypes=['folder', 'file']))

def size_of(entity_id):
    bundle = syn.restPOST(f'/entity/{entity_id}/bundle2', json.dumps({'includeFileHandles': True}))
    return bundle['fileHandles'][0]['contentSize']

def summarize(folder, prefix):
    parts = sorted((c for c in children(folder) if c['name'].startswith(prefix)), key=lambda c: c['name'])
    sizes = [size_of(p['id']) for p in parts]
    print(f'{prefix}: {len(parts)} partes, {sum(sizes) / 1e9:.1f} GB')
    return parts, sizes

train_parts, train_sizes = summarize(TRAIN_FOLDER, 'ChallengeData.tar.gz')
val_parts, val_sizes = summarize(AFTER_FOLDER, 'ChallengeData_validation')
test_parts, test_sizes = summarize(AFTER_FOLDER, 'ChallengeData_test')

# documentação do dataset
docs_dir = os.path.join(DEST, 'docs')
for c in children(TRAIN_FOLDER) + children(AFTER_FOLDER):
    if c['type'].endswith('Folder') and c['name'] == 'text-files':
        for t in children(c['id']):
            syn.get(t['id'], downloadLocation=docs_dir, ifcollision='overwrite.local')
    elif c['name'].endswith('.md'):
        syn.get(c['id'], downloadLocation=os.path.join(docs_dir, c['name'] + '_' + c['id']), ifcollision='overwrite.local')
for root, _, files in os.walk(docs_dir):
    for f in sorted(files):
        print('=' * 20, os.path.join(root, f))
        print(open(os.path.join(root, f), errors='replace').read()[:4000])

## 4. Baixar as partes do conjunto de treino para o Drive

O conjunto de treino é o que tem as segmentações. Partes que já estão completas no Drive são puladas, então dá para rodar várias vezes.

In [ ]:
def download(parts, sizes):
    for p, size in zip(parts, sizes):
        path = os.path.join(PARTS_DIR, p['name'])
        if os.path.exists(path) and os.path.getsize(path) == size:
            print('já existe', p['name'])
            continue
        print(f"baixando {p['name']} ({size / 1e9:.1f} GB)")
        syn.get(p['id'], downloadLocation=PARTS_DIR, ifcollision='overwrite.local')

download(train_parts, train_sizes)

## 5. Ver o que existe dentro do arquivo

As partes são pedaços de **um único** `.tar.gz`. Esta célula lê todas em sequência (sem juntar num arquivo gigante) e salva a lista de arquivos internos em `contents_train.txt`. Ela percorre o arquivo inteiro, então demora.

In [ ]:
import io, tarfile

class ConcatParts(io.RawIOBase):
    """Lê vários arquivos em sequência como se fossem um só (stream)."""
    def __init__(self, paths):
        self.paths, self.f = list(paths), None
    def readable(self):
        return True
    def readinto(self, buf):
        while True:
            if self.f is None:
                if not self.paths:
                    return 0
                self.f = open(self.paths.pop(0), 'rb')
            n = self.f.readinto(buf)
            if n:
                return n
            self.f.close()
            self.f = None

def part_paths(prefix):
    return sorted(os.path.join(PARTS_DIR, f) for f in os.listdir(PARTS_DIR) if f.startswith(prefix))

def open_stream(prefix):
    raw = io.BufferedReader(ConcatParts(part_paths(prefix)), buffer_size=16 * 1024 * 1024)
    return tarfile.open(fileobj=raw, mode='r|gz')

listing = os.path.join(DEST, 'contents_train.txt')
with open_stream('ChallengeData.tar.gz') as tar, open(listing, 'w') as out:
    for m in tar:
        out.write(f'{m.size}\t{m.name}\n')
print('lista salva em', listing)

# resumo: tamanho por pasta de 3 níveis
from collections import Counter
total = Counter()
for line in open(listing):
    size, name = line.rstrip('\n').split('\t', 1)
    total['/'.join(name.split('/')[:4])] += int(size)
for k, v in sorted(total.items(), key=lambda kv: -kv[1])[:40]:
    print(f'{v / 1e9:8.2f} GB  {k}')

## 6. Extrair só o necessário para o DeepMesh

Ajuste `KEEP` olhando a lista da célula anterior. O padrão guarda:
* imagens/segmentações em NIfTI (`.nii`, `.nii.gz`) — as segmentações guardam as coordenadas originais, que o DeepMesh usa para relacionar SAX e LAX;
* o k-space **totalmente amostrado** de cine **single-coil** (vira imagem com uma IFFT), se não houver imagens prontas;
* arquivos de texto/planilhas.

Tudo que é acelerado (`AccFactor`) ou multi-coil é descartado.

In [ ]:
import fnmatch

KEEP = ['*.nii', '*.nii.gz', '*.txt', '*.csv', '*.md', '*.json',
        '*SingleCoil*Cine*FullSample*']
DROP = ['*AccFactor*', '*MultiCoil*']

def wanted(name):
    return any(fnmatch.fnmatch(name, k) for k in KEEP) and not any(fnmatch.fnmatch(name, d) for d in DROP)

kept, kept_bytes = 0, 0
with open_stream('ChallengeData.tar.gz') as tar:
    for m in tar:
        if m.isfile() and wanted(m.name):
            target = os.path.join(OUT_DIR, m.name)
            if os.path.exists(target) and os.path.getsize(target) == m.size:
                continue
            tar.extract(m, OUT_DIR, filter='data')  # impede caminhos fora de OUT_DIR
            kept += 1
            kept_bytes += m.size
print(f'{kept} arquivos extraídos ({kept_bytes / 1e9:.1f} GB) em {OUT_DIR}')

## 7. (Opcional) Liberar espaço

Depois de conferir a extração, as partes `.tar.gz` podem ser apagadas do Drive. **Só rode se a pasta `extracted` estiver completa.**

In [ ]:
# import shutil; shutil.rmtree(PARTS_DIR)